# Module 2 — Mô hình Poisson Regression (Cửa hàng × Nhóm hàng × Ngày)

Dự báo `so_luong` mỗi ngày cho từng cặp cửa hàng × nhóm hàng bằng **hồi quy Poisson** — mô hình tuyến tính tổng quát (GLM) với hàm liên kết log, dành cho dữ liệu đếm: log E[y] = β₀ + Σ βⱼxⱼ, nên dự báo luôn ≥ 0. Phạt L2 (`alpha`) giữ hệ số ổn định khi các đặc trưng lag/rolling tương quan mạnh.

- **Đầu vào:** `outputs/daily_store_category.csv` (từ `module3/Module3_1_tien_xu_ly.ipynb`) + đặc trưng `tao_dac_trung` của Module 3.
- **So sánh công bằng:** cùng bảng, cùng `FEATS`, cùng mốc chia (`chia_tap`), cùng chỉ số (`danh_gia`) và baseline (`bang_baseline`) với các mô hình khác.
- **Xử lý riêng của mô hình tuyến tính** (cùng các cột, không thêm thông tin mới): `thu` và ba cột mã (`Mã cửa hàng_code`, `Nhóm hàng_code`, `Vùng_code`) được mã hoá one-hot thay vì coi là số thứ tự; các đặc trưng lượng bán gần đây có thể lấy `log1p` (chọn trên Validation); mọi cột số được chuẩn hoá.
- **Đầu ra** `outputs/poisson/`: `poisson_tuning.csv`, `poisson_best_params.json`, `poisson_metrics.csv`, `poisson_model.joblib`, `poisson_predictions.csv`, `poisson_feature_importance.*`, `poisson_error_by_category.csv`, `poisson_error_by_store.csv`, `poisson_thuc_te_vs_du_bao.png`.

**Cấu hình** (`.env` ở gốc repo, mẫu `.env.example`). **Chạy trên Google Colab:** upload `module3_preprocess.py` và `.env` cùng thư mục với notebook, rồi Runtime → Run all.

In [ ]:
import os, sys, json, time, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import PoissonRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.exceptions import ConvergenceWarning

sys.path.append("../module3")             # module tiền xử lý dùng chung của Module 3
from module3_preprocess import (OUT_DIR as OUT, SEED, doc_bang_ngay, tao_dac_trung, chia_tap, danh_gia,
                                bang_baseline, loi_theo_nhom, nop_ket_qua, tai_ve)

pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
OUT_PO = f"{OUT}/poisson"; os.makedirs(OUT_PO, exist_ok=True)
daily = doc_bang_ngay()
print(f"Bảng mô hình: {len(daily):,} dòng")

## 1. Đặc trưng và chia tập

Dùng nguyên `FEATS` của Module 3, chia thành 3 nhóm để tiền xử lý cho mô hình tuyến tính:

- **Lượng bán gần đây** (`lag_1/7/14`, `tb_7/14`, `std_7/14`): chuẩn hoá; có thể lấy `log1p` trước (chọn ở mục 2). Với hàm liên kết log, dùng thẳng số lượng làm biến đầu vào nghĩa là mỗi sản phẩm bán thêm hôm qua nhân dự báo lên một tỷ lệ cố định — dễ phóng đại ở các ô bán buôn lớn; `log1p` biến quan hệ này thành dạng luỹ thừa, ổn định hơn.
- **Lịch dạng số** (`cuoi_tuan`, `ngay_trong_thang`, `ngay_toi_tet`): chuẩn hoá.
- **Phân loại** (`thu`, `Mã cửa hàng_code`, `Nhóm hàng_code`, `Vùng_code`): one-hot — mã cửa hàng 0…129 không có thứ tự nên không thể dùng như một số.

In [ ]:
d, FEATS = tao_dac_trung(daily)
SO = ["lag_1", "lag_7", "lag_14", "tb_7", "tb_14", "std_7", "std_14"]
LICH = ["cuoi_tuan", "ngay_trong_thang", "ngay_toi_tet"]
PL = ["thu", "Mã cửa hàng_code", "Nhóm hàng_code", "Vùng_code"]
assert sorted(SO + LICH + PL) == sorted(FEATS)          # dùng đúng và đủ FEATS chung
fit, val, tr, te = chia_tap(d)

base_val = pd.DataFrame(bang_baseline(val)).T
rows = bang_baseline(te)
print("Baseline trên Validation:"); display(base_val)
print("Baseline trên Test:"); display(pd.DataFrame(rows).T)

## 2. Tối ưu tham số trên Validation

- Lưới 12 cấu hình: `alpha` (mức phạt L2) ∈ {10⁻⁵, 10⁻⁴, 10⁻³, 10⁻², 10⁻¹, 1} × biến đổi lượng bán ∈ {`log1p`, `goc`}.
- Train trên 15/01–14/02, chấm trên Validation 15–28/02; **chọn cấu hình theo MAE trên Validation** — cùng tiêu chí với các mô hình khác.
- Cấu hình chưa hội tụ sau `max_iter` vòng được ghi lại ở cột `hoi_tu`.

In [ ]:
MAX_ITER = 2000

def dung_mo_hinh(alpha, bien_doi):
    """Pipeline: tiền xử lý theo nhóm cột → PoissonRegressor (liên kết log, phạt L2)."""
    so = make_pipeline(FunctionTransformer(np.log1p, feature_names_out="one-to-one"), StandardScaler()) if bien_doi == "log1p" else StandardScaler()
    pre = ColumnTransformer([("so", so, SO), ("lich", StandardScaler(), LICH),
                             ("pl", OneHotEncoder(handle_unknown="ignore", sparse_output=False), PL)])
    return Pipeline([("pre", pre), ("m", PoissonRegressor(alpha=alpha, max_iter=MAX_ITER))])

params = [{"alpha": a, "bien_doi": b} for b in ["log1p", "goc"] for a in [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0]]
log = []
for i, p in enumerate(params):
    t = time.time()
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always", ConvergenceWarning)
        m = dung_mo_hinh(**p).fit(fit[FEATS], fit["so_luong"])
    du_bao = m.predict(val[FEATS])
    s = danh_gia(val["so_luong"], np.clip(du_bao, 0, None)) if np.isfinite(du_bao).all() else \
        dict.fromkeys(["MAE", "RMSE", "WAPE", "R2", "Bias"], np.inf)       # tràn số mũ → loại cấu hình
    hoi_tu = not any(issubclass(x.category, ConvergenceWarning) for x in w)
    log.append({"id": i, **p, "hoi_tu": hoi_tu, **s, "giây": round(time.time() - t, 1)})
    print(f"{i + 1:2}/{len(params)} MAE={s['MAE']:.4f} WAPE={s['WAPE']:.3f} Bias={s['Bias']:+.1%} {p}"
          + ("" if hoi_tu else " (chưa hội tụ)"))
tuning = pd.DataFrame(log).sort_values("MAE").reset_index(drop=True)
tuning.to_csv(f"{OUT_PO}/poisson_tuning.csv", index=False, encoding="utf-8-sig")
best = params[tuning.loc[0, "id"]]
with open(f"{OUT_PO}/poisson_best_params.json", "w") as f:
    json.dump({**best, "max_iter": MAX_ITER, "one_hot": PL, "chuan_hoa": SO + LICH}, f, indent=2, ensure_ascii=False)
print("Tốt nhất:", best)
display(tuning.head(5))

## 3. Train lại trên toàn bộ Train và chấm Test

Train lại cấu hình tốt nhất trên 15/01–28/02 rồi chấm trên tháng 3. Dự báo Poisson luôn ≥ 0; vẫn cắt về 0 cho đúng quy ước chung.

In [ ]:
po = dung_mo_hinh(**best).fit(tr[FEATS], tr["so_luong"])
te["du_bao_poisson"] = np.clip(po.predict(te[FEATS]), 0, None)
rows["Poisson"] = danh_gia(te["so_luong"], te["du_bao_poisson"])
result = pd.DataFrame(rows).T
result["WAPE so với MA7"] = result["WAPE"] / result.loc["MA7", "WAPE"] - 1
result.to_csv(f"{OUT_PO}/poisson_metrics.csv", encoding="utf-8-sig")
joblib.dump(po, f"{OUT_PO}/poisson_model.joblib", compress=3)
display(result)

## 4. Đặc trưng quan trọng và lỗi theo nhóm hàng / cửa hàng / ngày

**Mức quan trọng** theo chuẩn chung cho Ridge/Poisson là trị tuyệt đối hệ số trên đặc trưng đã chuẩn hoá. Với cột one-hot (một đặc trưng gốc thành nhiều cột 0/1) dùng cách tổng quát hoá tương đương: **độ lệch chuẩn phần đóng góp của đặc trưng vào log E[y]** trên tập Train — với cột số đã chuẩn hoá, giá trị này đúng bằng |β|.

**Kiểm tra giả định Poisson:** mô hình giả định phương sai = trung bình. Tỷ số Pearson χ²/bậc tự do của phần dư ≈ 1 nếu giả định đúng; lớn hơn nhiều là **phân tán quá mức** (overdispersion).

In [ ]:
# đóng góp của từng đặc trưng gốc vào bộ dự báo tuyến tính log E[y]
pre, glm = po.named_steps["pre"], po.named_steps["m"]
X = pre.transform(tr[FEATS]); ten_cot = pre.get_feature_names_out()
nguon = [next(f for f in FEATS if c.split("__", 1)[1] == f or c.split("__", 1)[1].startswith(f + "_")) for c in ten_cot]
dong_gop = pd.DataFrame(X * glm.coef_, columns=ten_cot).T.groupby(nguon).sum().T
imp = dong_gop.std().reindex(FEATS).sort_values()
imp.to_csv(f"{OUT_PO}/poisson_feature_importance.csv", header=["importance"], encoding="utf-8-sig")
print("Hệ số trên đặc trưng số đã chuẩn hoá (dấu cho biết chiều tác động):")
display(pd.Series(glm.coef_[: len(SO + LICH)], index=SO + LICH).round(3).to_frame("hệ số"))
fig, ax = plt.subplots(figsize=(7, 5)); ax.barh(imp.index, imp.values, color="#eb6834")
ax.set_title("Poisson — mức độ quan trọng của đặc trưng"); fig.tight_layout()
fig.savefig(f"{OUT_PO}/poisson_feature_importance.png", dpi=150); plt.show()

# giả định Poisson: Pearson chi² / bậc tự do của phần dư trên Train
mu = po.predict(tr[FEATS])
phan_tan = (((tr["so_luong"] - mu) ** 2 / mu).sum()) / (len(tr) - X.shape[1] - 1)
print(f"Pearson χ²/bậc tự do = {phan_tan:,.2f} (≈ 1 nếu đúng giả định Poisson)")
print(f"Phương sai / trung bình của so_luong trên Train = {tr['so_luong'].var() / tr['so_luong'].mean():,.1f}")

err_cat = loi_theo_nhom(te, "du_bao_poisson", "Nhóm hàng")
err_store = loi_theo_nhom(te, "du_bao_poisson", ["Mã cửa hàng", "Khu vực"])
err_cat.to_csv(f"{OUT_PO}/poisson_error_by_category.csv", encoding="utf-8-sig")
err_store.to_csv(f"{OUT_PO}/poisson_error_by_store.csv", encoding="utf-8-sig")
display(err_cat)
print(f"Poisson có WAPE thấp hơn MA7 ở {(err_cat.WAPE < err_cat.WAPE_MA7).sum()}/{len(err_cat)} nhóm hàng "
      f"và {(err_store.WAPE < err_store.WAPE_MA7).sum()}/{len(err_store)} cửa hàng")

day = te.groupby("Ngày")[["so_luong", "du_bao_poisson", "tb_7"]].sum()
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(day.index, day["so_luong"], color="#1f2a2a", lw=2, label="Thực tế")
ax.plot(day.index, day["du_bao_poisson"], color="#eb6834", lw=2, label="Poisson")
ax.plot(day.index, day["tb_7"], color="#898781", lw=1.5, ls="--", label="MA7")
ax.set_title("Tổng số lượng tháng 3 — thực tế và dự báo"); ax.legend(); ax.grid(axis="y", alpha=.3)
fig.tight_layout(); fig.savefig(f"{OUT_PO}/poisson_thuc_te_vs_du_bao.png", dpi=150); plt.show()
te[["Mã cửa hàng", "Nhóm hàng", "Khu vực", "Vùng", "Ngày", "so_luong", "du_bao_poisson", "lag_1", "lag_7", "tb_7"]] \
    .to_csv(f"{OUT_PO}/poisson_predictions.csv", index=False, encoding="utf-8-sig")

## 5. Kết quả và thảo luận

*Số liệu từ lần chạy ngày 10/10/2026 trên bảng mô hình chưa lọc (giữ bán buôn, online, trả lại) — cùng bảng với Random Forest và LightGBM; Test tháng 3 gồm 67.146 ô. Chạy lại sau khi nhóm chốt bộ lọc thì cập nhật lại mục này.*

**Tối ưu tham số.** Cấu hình tốt nhất trên Validation: `alpha=1e-05`, đặc trưng lượng bán lấy `log1p` — MAE Validation 0,771, **tốt hơn MA7 (0,847)**. Biến đổi `log1p` là lựa chọn quan trọng nhất: cả 6 cấu hình `log1p` có `alpha` ≤ 10⁻² đều tốt hơn cấu hình `goc` cùng `alpha`; phạt mạnh (`alpha` ≥ 0,1) làm MAE tăng nhanh. Cả 3 cấu hình tốt nhất đều dự báo thiếu khoảng 27–29% tổng lượng trên Validation.

| Cấu hình (3 tốt nhất theo MAE Val) | MAE Val | RMSE Val | WAPE Val | Bias Val |
|---|---:|---:|---:|---:|
| alpha 1e-05 · log1p | **0,771** | **8,167** | **0,682** | −29,4% |
| alpha 1e-04 · log1p | 0,772 | 8,168 | 0,683 | −29,2% |
| alpha 1e-03 · log1p | 0,777 | 8,182 | 0,687 | −27,3% |
| *MA7 (tham chiếu)* | *0,847* | *8,474* | *0,749* | *+11,2%* |

**Kết quả trên Test (tháng 3, 67.146 ô):**

| Mô hình | MAE | RMSE | WAPE | R² | Bias |
|---|---:|---:|---:|---:|---:|
| Naive (lag 1) | 0,887 | 6,675 | 0,802 | −0,142 | −0,9% |
| Seasonal Naive (lag 7) | 0,908 | 6,728 | 0,821 | −0,160 | +1,6% |
| MA7 | **0,744** | **4,975** | **0,674** | **0,365** | **−0,2%** |
| **Poisson** | 0,799 | 5,196 | 0,723 | 0,308 | +5,7% |

**Thảo luận.**

1. **Poisson thua MA7 trên mọi chỉ số Test** — MAE 0,799 so với 0,744, WAPE kém 7,3% (0,723 so với 0,674), RMSE 5,196 so với 4,975, R² 0,308 so với 0,365. Đáng chú ý là trên Validation Poisson lại thắng MA7 rõ (MAE 0,771 so với 0,847): Validation 15–28/02 rơi vào giai đoạn phục hồi sau Tết nên không đại diện cho tháng 3.
2. **Tổng lượng gần đúng nhưng lệch theo thời gian trong tháng.** Bias +5,7%: trung bình 2.530 sản phẩm/ngày toàn hệ thống so với thực tế 2.393 (MA7 2.388). Phần lệch không đều: dư 19,2% ở 01–10/03, dư 7,5% ở 11–20/03, thiếu 11,7% ở 21–31/03. Nguyên nhân là `ngay_trong_thang` có hệ số −0,191 (trên đặc trưng chuẩn hoá): mô hình học từ tháng 1, khi các ngày cuối tháng trùng Tết và gần như không bán, rồi áp quy luật "cuối tháng bán ít" sang tháng 3. Ở 278 ô bán ≥ 20 sản phẩm/ngày (thực tế 13.570), Poisson dự báo 12.695 — gần thực tế nhất trong các mô hình đã có (MA7 10.904, Random Forest 10.622, LightGBM 4.589); ngược lại ở 46.477 ô bán 0, Poisson dự báo tổng 12.224 (MA7 11.195), MAE 0,263 so với 0,241.
3. **Thắng MA7 ở 6/23 nhóm hàng nhưng chỉ chiếm 3,8% lượng bán** (Laptop, Sim thẻ, Khuyến mãi, Phần mềm, Linh kiện máy tính, Điện lạnh), và 33/130 cửa hàng. Thua ở cả ba nhóm lớn: Điện thoại (WAPE 0,600 so với 0,559), Phụ kiện (0,556 so với 0,507), Hcare (0,584 so với 0,552).
4. **Đặc trưng quan trọng:** nhóm hàng (43,6%), mã cửa hàng (10,3%), vùng (8,8%), lag 1 (7,9%), ngày trong tháng (7,2%). Ba cột mã cộng lại 62,7%, lượng bán gần đây (lag, trung bình, độ lệch chuẩn) 24,9%, lịch 12,3%. Mô hình tuyến tính trên thang log đặt một mức bán cố định cho mỗi nhóm hàng và cửa hàng, rồi chỉ điều chỉnh quanh mức đó theo lượng bán gần đây — khác Random Forest (96% dựa vào lượng bán gần đây). Lưu ý: mức quan trọng của Poisson (độ lệch chuẩn phần đóng góp vào log E[y]) và của mô hình cây đo theo cách khác nhau nên chỉ so sánh được thứ tự, không so sánh trực tiếp con số.
5. **Giả định Poisson bị vi phạm.** Trên Train, phương sai của `so_luong` gấp 53,2 lần trung bình; sau khi mô hình đã giải thích bằng các đặc trưng, tỷ số Pearson χ²/bậc tự do vẫn là 2,31 (≈ 1 nếu đúng giả định). Dữ liệu còn **phân tán quá mức**, đúng như Kolassa (2016) và Ziel (2022) mô tả cho dữ liệu bán lẻ: dự báo điểm vẫn dùng được nhưng mô hình đánh giá thấp độ biến động thực tế.
6. **So với các mô hình đã có** (cùng bảng, đặc trưng, mốc chia): Poisson có MAE cao nhất (0,799; Random Forest 0,757, LightGBM 0,678). LightGBM thắng MA7 về MAE/WAPE nhưng dự báo thiếu 37,3% tổng lượng. Như Random Forest, Poisson đạt điều kiện |Bias| ≤ 10% (+5,7%) và dự báo tốt các ô bán lớn, nhưng theo tiêu chí đề xuất (MAE thấp nhất trong các mô hình có |Bias| ≤ 10%) Poisson đứng sau Random Forest, và cả hai vẫn chưa vượt MA7.

**Hướng cải thiện:** thay `ngay_trong_thang` bằng cờ giai đoạn Tết (cần Module 4 chốt vì `FEATS` dùng chung); thử hồi quy Negative Binomial hoặc Tweedie để xử lý phân tán quá mức.

**Giới hạn:** bảng mô hình chưa lọc bán buôn/online; mô hình tuyến tính trên thang log không học được tương tác (vd hiệu ứng cuối tuần khác nhau giữa nhóm hàng); một mốc chia; Validation không đại diện cho Test (Bias −29,4% trên Validation so với +5,7% trên Test).

## Nộp kết quả vào báo cáo và tải về (Colab)

`nop_ket_qua` lọc các kết quả đưa vào báo cáo sang `bao_cao/du_lieu/poisson/`. Sau đó chạy `python module5/tao_bao_cao.py` để cập nhật `bao_cao/BAO_CAO.md` rồi commit `bao_cao/` (trên Colab: giải nén `bao_cao_poisson.zip` vào `bao_cao/du_lieu/poisson/` trong repo trước).

In [ ]:
nop_ket_qua("poisson", val)
tai_ve(OUT_PO, "module2_poisson_outputs")